### Packages

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

Gephi clustering settings:
- import graph object
- Statistics: Connected Components: Run (standard settings)
- Statistics: Modularity: Run (standard settings)
- Layout: Circle Pack Layout: Hierarchy1 = Component ID, Hierarchy2 = Modularity Class

### Load data

Load all epitope obs dfs

In [2]:
from pathlib import Path

folder = Path("./sc_obs_files/naive")

df_list = [
    pd.read_csv(file)
    for file in folder.glob("*.csv")
]
#qc_file = pd.read_excel('./ref_tcr_data/Library_QC.xlsx')

In [3]:
df_obs_new = pd.concat(df_list, ignore_index=True)

In [4]:
df_obs_new['donor_id'].nunique()

10

In [5]:
def process_df_obs(df):
    df = df.copy()
    # Rename TRBV10 -> TRBV4 wrong annotation by parse/cellranger
    df = df.replace("TRBV10", "TRBV4", regex=False)
    # Remove unwanted V genes: non-functional
    df = df[
        ~df["IR_VDJ_1_v_call"].isin(["TRBV21", "TRBV8"])
    ]
    # Keep only desired chain pairings - only single chain TCRs are considered
    df = df[
        df["chain_pairing"].isin([
            "single pair",
            "assigned_single_pair"
        ])
    ]
    return df.reset_index(drop=True).copy()

In [6]:
df_obs_new = process_df_obs(df_obs_new)
df_obs_new['tcr_id'] = df_obs_new['TCR']
df_obs_new['clone_size_mouse'] = df_obs_new['TCR_expansion']

In [7]:
#qc_list = qc_file.tcr_id.tolist()
#df_obs_new = df_obs_new[~df_obs_new.tcr_id.isin(qc_list)].reset_index(drop=True).copy()

In [8]:
df_obs_new['donor_id'].nunique()

10

## QC sequencing specificity annotation

Multi specificity annotations per tcr id are only epitope + below_threshold -> consistent annotation across experiments

Annotate sharedness

In [9]:
df_obs_new["shared_mouse"] = (
    df_obs_new.groupby("tcr_id")["donor_id"]
      .transform("nunique")
)

In [10]:
cols_to_keep = ['is_cell',
 'high_confidence',
 'multi_chain',
 'extra_chains',
 'IR_VJ_1_c_call',
 'IR_VDJ_1_c_call',
 'IR_VJ_1_d_call',
 'IR_VDJ_1_d_call',
 'IR_VJ_1_j_call',
 'IR_VDJ_1_j_call',
 'IR_VJ_1_junction',
 'IR_VDJ_1_junction',
 'IR_VJ_1_junction_aa',
 'IR_VDJ_1_junction_aa',
 'IR_VJ_1_v_call',
 'IR_VDJ_1_v_call',
 'has_ir',
 'experiment',
 'group',
 'batch',
 'mouse',
 'mouse_concat',
 'isolated_specificity',
 'receptor_type',
 'receptor_subtype',
 'chain_pairing',
 'tra_id',
 'trb_id',
 'tcr_id',
 'TCR_expansion',
 'clonotype_size_mouse',
 'clone_size_mouse',
 'sequencing_specificity',
 'shared_mouse',
 'NAME',
 'donor_id',
 'biosample_id',
 'species',
 'species__strain',
 'species__ontology_label',
 'disease',
 'disease__strain',
 'disease__ontology_label',
 'disease__epitope',
 'disease__antigen',
 'organ',
 'organ__ontology_label',
 'library_preparation_protocol',
 'library_preparation_protocol__ontology_label',
 'cell_type',
 'cell_type__ontology_label',
 'sex',
 'assigned_TRA',
 'assigned_TRB',
 'TRB',
 'TRA',
 'TCR',
 'TCR_expansion_full_data',
 'n_cells_mouse',
 'cell_id']

In [11]:
df_obs_new = df_obs_new[cols_to_keep].copy()


### Make TCR data sheet

In [12]:
v_info = pd.read_csv('./10X_mouse_vdj/cellranger_mouse_v_ref.csv')
j_info = pd.read_csv('./10X_mouse_vdj/cellranger_mouse_j_ref.csv')

In [13]:
df = df_obs_new.rename(columns={
    "IR_VJ_1_junction_aa": "cdr3a_aa",
    "IR_VDJ_1_junction_aa": "cdr3b_aa",
    "IR_VJ_1_junction": "cdr3a_nt",
    "IR_VDJ_1_junction": "cdr3b_nt",
    "IR_VJ_1_v_call": "va",
    "IR_VJ_1_j_call": "ja",
    "IR_VJ_1_c_call": "ca",
    "IR_VDJ_1_v_call": "vb",
    "IR_VDJ_1_j_call": "jb",
    "IR_VDJ_1_d_call": "db",
    "IR_VDJ_1_c_call": "cb",
})

In [14]:
from preprocess_tcr_2 import stitch_nt_from_cellranger
from pandas.api.types import is_numeric_dtype

agg_dict = {}

for col in df.columns:
    #if col == "tcr_id":
    #    continue

    if is_numeric_dtype(df[col]):
        agg_dict[col] = "max"
    else:
        agg_dict[col] = "first"

df = (
    df.groupby("tcr_id", as_index=False, observed=True)
      .agg(agg_dict)
)
    
df = df.reset_index(drop=True)
df = stitch_nt_from_cellranger(
        df=df,
        v_info=v_info,
        j_info=j_info,
        v_gene_col="vb",
        j_gene_col="jb",
        cdr3_col="cdr3b_nt",
        c_col="cb",
        chain="beta"
        )
    
df = stitch_nt_from_cellranger(
        df=df,
        v_info=v_info,
        j_info=j_info,
        v_gene_col="va",
        j_gene_col="ja",
        cdr3_col="cdr3a_nt",
        c_col="ca",
        chain="alpha"
        )

df = df.reset_index(drop=True)
  

C:\Users\wwspa\miniconda3\envs\tcr_scripts\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [15]:
df['annotated_specificity'] = 'naive'

In [16]:
df.columns

Index(['is_cell', 'high_confidence', 'multi_chain', 'extra_chains', 'ca', 'cb',
       'IR_VJ_1_d_call', 'db', 'ja', 'jb', 'cdr3a_nt', 'cdr3b_nt', 'cdr3a_aa',
       'cdr3b_aa', 'va', 'vb', 'has_ir', 'experiment', 'group', 'batch',
       'mouse', 'mouse_concat', 'isolated_specificity', 'receptor_type',
       'receptor_subtype', 'chain_pairing', 'tra_id', 'trb_id', 'tcr_id',
       'TCR_expansion', 'clonotype_size_mouse', 'clone_size_mouse',
       'sequencing_specificity', 'shared_mouse', 'NAME', 'donor_id',
       'biosample_id', 'species', 'species__strain', 'species__ontology_label',
       'disease', 'disease__strain', 'disease__ontology_label',
       'disease__epitope', 'disease__antigen', 'organ',
       'organ__ontology_label', 'library_preparation_protocol',
       'library_preparation_protocol__ontology_label', 'cell_type',
       'cell_type__ontology_label', 'sex', 'assigned_TRA', 'assigned_TRB',
       'TRB', 'TRA', 'TCR', 'TCR_expansion_full_data', 'n_cells_mouse',
     

## Annotate pgen

In [17]:
from preprocess_tcr_2 import format_for_tcrdist

df['clonotype_origin'] = df['tcr_id']
df_dist = format_for_tcrdist(df)

from preprocess_tcr_2 import run_tcrdist_pgen_only

# Assume you already have:
#   df_clonotypes_all  (output of the previous function)
#   df_cells           (original per-cell DataFrame)
#   metrics_a, metrics_b, weights_a, weights_b, kargs_a, kargs_b
result_list = []

results = run_tcrdist_pgen_only(
        df_clonotypes_all = df_dist, ### dataframe for TCRdist calculation with formatting of v genes and clone id and so on
        df_original       = df, ### dataframe before it was processed for TCRdist calculation
        db_file           = "alphabeta_gammadelta_db.tsv",
        olga_beta_folder  = "mouse_T_beta",
        olga_alpha_folder = "mouse_T_alpha",
        cpus              = 12
    )

Generate new TR file
calculating TCR pgens


83352it [02:43, 509.72it/s]                                                                                            
83352it [02:10, 637.00it/s]                                                                                            


Merging data


In [18]:
df_pgen = results[0].copy()
df_pgen['tcr_id'] = df_pgen['clonotype_origin']
df["pgen_cdr3_b_aa"] = (
    df["tcr_id"]
    .map(
        df_pgen.set_index("tcr_id")["pgen_cdr3_b_aa"]
    )
)
df["pgen_cdr3_a_aa"] = (
    df["tcr_id"]
    .map(
        df_pgen.set_index("tcr_id")["pgen_cdr3_a_aa"]
    )
)
df["pgen_tcr"] = df["pgen_cdr3_a_aa"] * df["pgen_cdr3_b_aa"]

## Annotate obs df

Export TCR data sheet

In [19]:
df_backup = df.copy()

In [21]:
df.to_excel('./processed_tcr_data_new_sc_processing/TCR_data_busch_lab_naive_all_tcrs.xlsx')
df.to_csv('./processed_tcr_data_new_sc_processing/TCR_data_busch_lab_naive_all_tcrs.csv')

In [22]:
df_obs_new.to_excel('./processed_tcr_data_new_sc_processing/TCR_obs_data_busch_lab_naive_all_tcrs.xlsx')
df_obs_new.to_csv('./processed_tcr_data_new_sc_processing/TCR_obs_data_busch_lab_naive_all_tcrs.csv')